# 07. 세븐일레븐 키워드 풀 검토

**목적**: `smart_clean_result_seven_final.xlsx`의 `확정_키워드`를 3소스 확정 키워드 풀과 대조하여
풀에 없는 나머지 키워드를 식별합니다.

| 단계 | 내용 |
|---|---|
| Phase 1 | 3소스 parquet → `confirmed_pool` 구축 |
| Phase 2 | 세븐 `확정_키워드` × `confirmed_pool` 대조 |
| Phase 3 | 통계 + 제품별 분류 + 나머지 키워드 목록 |

In [1]:
import os, ast
import numpy as np
import pandas as pd

BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
PROC_DIR = os.path.join(BASE_DIR, 'data', 'processed')
EDA_DIR  = os.path.join(BASE_DIR, 'eda')

INSTA_PARQUET  = os.path.join(PROC_DIR, 'insta_keywords_processed.parquet')
BLOG_PARQUET   = os.path.join(PROC_DIR, 'blog_keywords_processed.parquet')
TREND_PARQUET  = os.path.join(PROC_DIR, 'trend_keywords_processed.parquet')
SEVEN_FINAL    = os.path.join(PROC_DIR, '편의점_instagram',
                              'smart_clean_result_seven_final.xlsx')

def safe_parse(val):
    """list / ndarray / str → list 변환. parquet에서 numpy array로 올 수 있음."""
    if isinstance(val, (list, np.ndarray)):
        return [str(x) for x in val]
    try:
        isna = pd.isna(val)
    except (ValueError, TypeError):
        isna = False
    if isna:
        return []
    s = str(val).strip()
    if s.startswith('['):
        try:
            return ast.literal_eval(s)
        except Exception:
            pass
    return [x.strip() for x in s.split(',') if x.strip()]

print('BASE_DIR:', BASE_DIR)

BASE_DIR: c:\Users\송정현\Documents\Projects\박재홍교수님세미나\Projects\20기\7eleven_npd_framework


In [2]:
import sys
import importlib
from collections import OrderedDict

# keyword_rules.py는 eda/ 디렉토리에 있음
EDA_SRC_DIR = os.path.join(BASE_DIR, 'eda')
if EDA_SRC_DIR not in sys.path:
    sys.path.insert(0, EDA_SRC_DIR)

import keyword_rules
importlib.reload(keyword_rules)

REMOVE_KWS        = set(keyword_rules.REMOVE_KWS)
SYNONYM_MAP       = dict(keyword_rules.SYNONYM_MAP)
SPLIT_MAP         = dict(keyword_rules.SPLIT_MAP)
SANDWICH_PRODUCTS = set(keyword_rules.SANDWICH_PRODUCTS)
CONTAINS_COLLAPSE = list(keyword_rules.CONTAINS_COLLAPSE)
EXTRA_SPLIT_MAP   = dict(keyword_rules.EXTRA_SPLIT_MAP)
EXTRA_SYNONYM_MAP = dict(keyword_rules.EXTRA_SYNONYM_MAP)
REMOVE_WORDS      = set(keyword_rules.REMOVE_WORDS)
KEYWORD_MAPPING   = dict(keyword_rules.KEYWORD_MAPPING)

# PROMO_MAP 역방향 병합: 키워드 → 프로모션 코드
for code, kws in keyword_rules.PROMO_MAP.items():
    for kw in kws:
        SYNONYM_MAP[kw] = code


def remove_noise_kws(kw_list):
    if not isinstance(kw_list, list):
        return kw_list
    return [kw for kw in kw_list if kw not in REMOVE_KWS]


def unify_keywords(kw_list):
    if not isinstance(kw_list, list):
        return kw_list
    result = []
    for kw in kw_list:
        if kw in SPLIT_MAP:
            result.extend(SPLIT_MAP[kw])
        else:
            result.append(SYNONYM_MAP.get(kw, kw))
    return list(dict.fromkeys(result))


def apply_advanced_rules(kw_list, product_name=''):
    if not isinstance(kw_list, list):
        return kw_list
    result = []
    for kw in kw_list:
        if kw == '제로':
            result.extend(
                ['제로', '카페인'] if product_name == '제로모히또제로카페인'
                else ['제로', '슈거']
            )
            continue
        if kw == '샌드':
            result.append('샌드위치' if product_name in SANDWICH_PRODUCTS else '샌드')
            continue
        if kw in EXTRA_SPLIT_MAP:
            result.extend(EXTRA_SPLIT_MAP[kw])
            continue
        collapsed = next((w for w in CONTAINS_COLLAPSE if w in kw), None)
        if collapsed:
            result.append(collapsed)
            continue
        if kw in REMOVE_WORDS:
            continue
        result.append(EXTRA_SYNONYM_MAP.get(kw, kw))
    return list(dict.fromkeys(result))


def apply_keyword_mapping(kw_list):
    if isinstance(kw_list, str):
        try:
            kw_list = ast.literal_eval(kw_list)
        except Exception:
            kw_list = [kw_list]
    if not isinstance(kw_list, list):
        return []
    expanded = []
    for kw in kw_list:
        expanded.extend(KEYWORD_MAPPING.get(kw, [kw]))
    return list(OrderedDict.fromkeys(expanded))


def run_pipeline(kw_list, product_name=''):
    kw_list = remove_noise_kws(kw_list)
    kw_list = unify_keywords(kw_list)
    kw_list = apply_advanced_rules(kw_list, product_name)
    kw_list = apply_keyword_mapping(kw_list)
    return kw_list


print('run_pipeline 로드 완료')
print(f'  SYNONYM_MAP : {len(SYNONYM_MAP):,}개 (PROMO_MAP 병합 포함)')
print(f'  REMOVE_KWS  : {len(REMOVE_KWS):,}개')

run_pipeline 로드 완료
  SYNONYM_MAP : 643개 (PROMO_MAP 병합 포함)
  REMOVE_KWS  : 164개


## Phase 1. 3소스 확정 키워드 풀 구축

`insta_keywords_processed` + `blog_keywords_processed` + `trend_keywords_processed`
각 파일의 `확정키워드_정제` 컬럼을 합산하여 `confirmed_pool` (set) 을 만듭니다.

In [3]:
confirmed_pool = set()

for label, path in [
    ('인스타', INSTA_PARQUET),
    ('블로그', BLOG_PARQUET),
    ('트렌드', TREND_PARQUET),
]:
    df = pd.read_parquet(path)
    col = '확정키워드_정제'
    if col not in df.columns:
        # fallback: 확정키워드 컬럼 사용
        col = [c for c in df.columns if '확정' in c and '키워드' in c][0]
    kws = df[col].apply(safe_parse).explode().dropna()
    kws = kws[kws.str.strip() != '']
    before = len(confirmed_pool)
    confirmed_pool.update(kws.tolist())
    print(f'[{label}] +{len(confirmed_pool) - before:,}개 → 누적 {len(confirmed_pool):,}개')

print(f'\n확정 키워드 풀 총 {len(confirmed_pool):,}개')

[인스타] +1,710개 → 누적 1,710개
[블로그] +824개 → 누적 2,534개
[트렌드] +139개 → 누적 2,673개

확정 키워드 풀 총 2,673개


## Phase 2. 세븐 확정_키워드 × confirmed_pool 대조

`smart_clean_result_seven_final.xlsx`의 `확정_키워드` 컬럼 기준.
각 제품별로 풀에 있는 키워드와 나머지(풀 외) 키워드를 분류합니다.

In [4]:
if not os.path.exists(SEVEN_FINAL):
    raise FileNotFoundError(
        f'파일 없음: {SEVEN_FINAL}\n'
        'smart_clean_result_seven.xlsx 검수 후 _final 이름으로 저장하세요.'
    )

df7 = pd.read_excel(SEVEN_FINAL)
print(f'세븐 검수 파일: {len(df7):,}개 제품')
print('컬럼:', df7.columns.tolist())

CONFIRMED_COL = '확정_키워드'
if CONFIRMED_COL not in df7.columns:
    raise KeyError(f'컬럼 없음: {CONFIRMED_COL} (실제 컬럼: {df7.columns.tolist()})')

rows = []
for _, row in df7.iterrows():
    name          = str(row['제품명'])
    confirmed_raw = safe_parse(row[CONFIRMED_COL])
    confirmed     = run_pipeline(confirmed_raw, product_name=name)  # 정규화 적용
    in_pool       = [kw for kw in confirmed if kw in confirmed_pool]
    remaining     = [kw for kw in confirmed if kw not in confirmed_pool]
    rows.append({
        '제품명':          name,
        '확정_키워드_raw': confirmed_raw,  # 원본 보존 (디버깅용)
        '확정_키워드':     confirmed,       # 정규화 후
        '풀_교집합':       in_pool,
        '나머지_키워드':   remaining,
        '총_키워드수':     len(confirmed),
        '풀_매칭수':       len(in_pool),
        '나머지수':        len(remaining),
    })

result_df = pd.DataFrame(rows)
print(f'\n분류 완료: {len(result_df):,}개 제품')

세븐 검수 파일: 859개 제품
컬럼: ['제품명', '정제_전_키워드', '확정_키워드', '생존후보_키워드']

분류 완료: 859개 제품


## Phase 3. 통계 및 검토

In [5]:
# 커버율 통계
total_kws    = result_df['총_키워드수'].sum()
pool_matched = result_df['풀_매칭수'].sum()
remaining_n  = result_df['나머지수'].sum()
products_with_remaining = (result_df['나머지수'] > 0).sum()

print('=== 전체 통계 ===')
print(f'전체 키워드:        {total_kws:,}개')
print(f'풀 매칭:            {pool_matched:,}개 ({pool_matched/total_kws*100:.1f}%)')
print(f'나머지(풀 외):      {remaining_n:,}개 ({remaining_n/total_kws*100:.1f}%)')
print(f'나머지 있는 제품:   {products_with_remaining:,}개 / {len(result_df):,}개')

=== 전체 통계 ===
전체 키워드:        4,798개
풀 매칭:            4,644개 (96.8%)
나머지(풀 외):      154개 (3.2%)
나머지 있는 제품:   131개 / 859개


In [6]:
# 나머지 키워드 있는 제품만 표시
review_df = result_df[result_df['나머지수'] > 0].sort_values('나머지수', ascending=False)
display_df = review_df[['제품명', '나머지_키워드', '나머지수', '총_키워드수']].reset_index(drop=True)
print(f'나머지 키워드 보유 제품: {len(display_df):,}개')
display_df

나머지 키워드 보유 제품: 131개


,제품명,나머지_키워드,나머지수,총_키워드수
0,더커진 맛다시 참치마요 에그말이,"[고추장, 마요네즈, 에그]",3,8
1,맛다시 참치마요 에그말이,"[고추장, 마요네즈, 에그]",3,8
2,맛장우맛자랑 직화닭갈비,"[기간한정, 봄맛페스티벌]",2,11
3,최강록의 로스팜에그샌드,"[마요네즈, 에그]",2,16
4,티처스 햄치즈에그팡팡토스트,"[에그, 티처스]",2,4
...,...,...,...,...
126,한도초과 폭풍햄가득참치김밥,[마요네즈],1,13
127,한입삼겹살,[집밥],1,11
128,화요 53도 적마에디션,[전통주],1,7
129,훈와리캬라멜,[주토피아],1,3


In [7]:
# 나머지 키워드 전체 목록 (unique, 빈도 내림차순)
from collections import Counter

all_remaining = []
for kws in result_df['나머지_키워드']:
    all_remaining.extend(kws)

remaining_counts = Counter(all_remaining)
remaining_series = pd.Series(remaining_counts, name='빈도').sort_values(ascending=False)
print(f'unique 나머지 키워드: {len(remaining_series):,}개')
remaining_series.head(50)

unique 나머지 키워드: 49개


마요네즈      29
고추장       13
에그        12
경기관람       6
슈거         6
봄맛페스티벌     4
아티제        4
패스츄리       4
집밥         4
티처스        4
청량감        3
스티커        3
케익         3
에너지충전      3
기간한정       3
통닭         3
흑미         2
출출할때       2
솔트         2
주토피아       2
피카츄        2
컵밥         2
반반         2
씨앗         2
땡초         2
해물         2
어니언        2
MZ세대       2
과실         2
한정판매       2
닭육수        2
무설탕        2
마이멜로디      2
김치찌개       1
떡국         1
게맛살        1
더블패티       1
우삼겹        1
베어스        1
멀티립        1
유니짜장       1
깔끔함        1
통밀         1
스모크햄       1
콜드브루       1
식빵         1
비타민C       1
하와이안       1
전통주        1
Name: 빈도, dtype: int64

## Phase 4. 나머지 키워드 수동 검토 시트 내보내기

`seven_remaining_kw_review.xlsx` 생성 → 열어서 `검토` 컬럼에 **O**(포함) / **X**(제외) 입력 후
파일명 뒤에 `_final` 붙여 저장 (`seven_remaining_kw_review_final.xlsx`).

In [8]:
REVIEW_OUT = os.path.join(EDA_DIR, 'seven_remaining_kw_review.xlsx')

review_sheet = (
    remaining_series
    .reset_index()
    .rename(columns={'index': '나머지_키워드'})
)
review_sheet.columns = ['나머지_키워드', '빈도']

# 유사 확정 키워드 추출 함수 정의
def get_similar_kws(kw):
    if not isinstance(kw, str) or not kw.strip():
        return ''
    kw_clean = kw.strip().lower()
    # confirmed_pool의 키워드 중, 현재 검토 중인 나머지_키워드를 포함하는 것들을 필터링
    matches = [c_kw for c_kw in confirmed_pool if kw_clean in str(c_kw).lower()]
    return ', '.join(sorted(matches))

review_sheet['유사_확정_키워드'] = review_sheet['나머지_키워드'].apply(get_similar_kws)
review_sheet['검토'] = ''  # O = pool inclusion / X = exclusion

# 열 순서 재조정
review_sheet = review_sheet[['나머지_키워드', '빈도', '유사_확정_키워드', '검토']]

review_sheet.to_excel(REVIEW_OUT, index=False)
print(f'검토 시트 저장: {REVIEW_OUT}')
print(f'총 {len(review_sheet):,}개 키워드')
print('→ 검토 컬럼에 O(포함) / X(제외) 입력 후 파일명에 _final 붙여 저장')

검토 시트 저장: c:\Users\송정현\Documents\Projects\박재홍교수님세미나\Projects\20기\7eleven_npd_framework\eda\seven_remaining_kw_review.xlsx
총 49개 키워드
→ 검토 컬럼에 O(포함) / X(제외) 입력 후 파일명에 _final 붙여 저장


In [9]:
query = '시험'
query_clean = str(query).strip().lower()
[kw for kw in confirmed_pool if query_clean in str(kw).lower()]

# 식사류 -> 식사
# 금계란 -> 계란
# 비건식단 -> 비건, 식단
# 통닭 - 치킨 통일
# 해물향 -> 해물
# 제로슈가 -> 제로, 설탕
# 매콤달콤

query = '에당'
result_df[result_df['나머지_키워드'].apply(lambda x: query in x)]

,제품명,확정_키워드_raw,확정_키워드,풀_교집합,나머지_키워드,총_키워드수,풀_매칭수,나머지수


## Phase 5. 검토 완료 키워드 풀 편입

`seven_remaining_kw_review_final.xlsx`의 `검토 == 'O'` 키워드를 `confirmed_pool`에 추가한 뒤
최종 커버율을 재계산합니다.

In [10]:
REVIEW_FINAL = os.path.join(EDA_DIR, 'seven_remaining_kw_review_final.xlsx')
IP_KW_PATH   = os.path.join(PROC_DIR, 'ip_keywords.parquet')

if not os.path.exists(REVIEW_FINAL):
    print(f'파일 없음: {REVIEW_FINAL}')
    print('Phase 4에서 내보낸 파일 검토 후 _final 붙여 저장하세요.')
else:
    df_reviewed = pd.read_excel(REVIEW_FINAL)

    # ── Step 3. 키워드 처리: action_map 구축 ──────────────────────────
    # 결정 컬럼: '유사_확정_키워드'
    #   빈 값  → [] 제거
    #   'O'    → [원래_키워드] 그대로 편입
    #   텍스트 → [kw1, kw2, ...] 콤마 분리 교체
    action_map = {}
    cnt_keep, cnt_replace, cnt_remove = 0, 0, 0

    for _, row in df_reviewed.iterrows():
        kw       = str(row['나머지_키워드']).strip()
        decision = str(row['유사_확정_키워드']).strip()

        if decision.lower() in ('', 'nan'):
            action_map[kw] = []
            cnt_remove += 1
        elif decision.upper() == 'O':
            action_map[kw] = [kw]
            cnt_keep += 1
        else:
            action_map[kw] = [r.strip() for r in decision.split(',') if r.strip()]
            cnt_replace += 1

    print('=== Step 3. 키워드 처리 결과 (unique 나머지 키워드 기준) ===')
    print(f'  O (그대로 편입) : {cnt_keep:,}개')
    print(f'  텍스트 교체     : {cnt_replace:,}개')
    print(f'  제거 (빈 값)    : {cnt_remove:,}개')

    # ── Step 4. 제품별 키워드에 action_map 반영 ───────────────────────
    rows2 = []
    n_kept_pool, n_kept_action, n_removed = 0, 0, 0

    for _, row in df7.iterrows():
        name          = str(row['제품명'])
        confirmed_raw = safe_parse(row[CONFIRMED_COL])
        confirmed     = run_pipeline(confirmed_raw, product_name=name)

        final_kws = []
        for kw in confirmed:
            if kw in confirmed_pool:
                final_kws.append(kw)
                n_kept_pool += 1
            elif kw in action_map:
                replacements = action_map[kw]
                final_kws.extend(replacements)
                n_kept_action += len(replacements)
                n_removed += (1 - min(len(replacements), 1))  # 제거된 경우만
            else:
                n_removed += 1  # action_map에도 없으면 제외

        final_kws = list(dict.fromkeys(final_kws))
        rows2.append({
            '제품명':    name,
            '최종키워드': final_kws,
            '키워드수':  len(final_kws),
        })

    final_df = pd.DataFrame(rows2)
    print(f'\n=== Step 4. 제품별 키워드 반영 완료 ===')
    print(f'총 {len(final_df):,}개 제품')

    # ── Step 5. 확장 풀 구축 ─────────────────────────────────────────
    extended_pool = confirmed_pool | set(
        kw for replacements in action_map.values() for kw in replacements
    )

    # ── Step 6. 최종 커버율 비교 ──────────────────────────────────────
    # Phase 2 기준 (정규화 후, 검수 전)
    p2_total   = result_df['총_키워드수'].sum()
    p2_matched = result_df['풀_매칭수'].sum()
    p2_remain  = result_df['나머지수'].sum()

    # Phase 5 기준 (검수 적용 후)
    p5_total   = final_df['키워드수'].sum()

    print(f'\n=== Step 6. 커버율 비교 ===')
    print(f'{'':30} {'Phase 2 (검수 전)':>20} {'Phase 5 (검수 후)':>20}')
    print(f'{'─'*72}')
    print(f'{'전체 키워드 수':30} {p2_total:>20,}개 {p5_total:>20,}개')
    print(f'{'풀 매칭 수':30} {p2_matched:>20,}개 {p5_total:>20,}개')
    print(f'{'커버율 (%)':30} {p2_matched/p2_total*100:>19.1f}% {"100.0":>19}%')
    print(f'{'나머지 (미처리)':30} {p2_remain:>20,}개 {"0":>20}개')
    print()
    print(f'  나머지 {p2_remain}개 처리 결과:')
    print(f'    편입 (O)  : {cnt_keep:,}개 unique ({n_kept_action:,}개 제품×키워드)')
    print(f'    교체      : {cnt_replace:,}개 unique')
    print(f'    제거      : {cnt_remove:,}개 unique')
    print(f'  확장 풀: 기존 {len(confirmed_pool):,}개 → {len(extended_pool):,}개 (+{len(extended_pool)-len(confirmed_pool):,}개)')

    no_kw = (~(final_df['키워드수'] > 0)).sum()
    print(f'\n  키워드 없는 제품: {no_kw:,}개 / 전체 {len(final_df):,}개')

=== Step 3. 키워드 처리 결과 (unique 나머지 키워드 기준) ===
  O (그대로 편입) : 72개
  텍스트 교체     : 132개
  제거 (빈 값)    : 58개

=== Step 4. 제품별 키워드 반영 완료 ===
총 859개 제품

=== Step 6. 커버율 비교 ===
                                     Phase 2 (검수 전)       Phase 5 (검수 후)
────────────────────────────────────────────────────────────────────────
전체 키워드 수                                      4,798개                4,816개
풀 매칭 수                                        4,644개                4,816개
커버율 (%)                                       96.8%               100.0%
나머지 (미처리)                                       154개                    0개

  나머지 154개 처리 결과:
    편입 (O)  : 72개 unique (182개 제품×키워드)
    교체      : 132개 unique
    제거      : 58개 unique
  확장 풀: 기존 2,673개 → 2,718개 (+45개)

  키워드 없는 제품: 12개 / 전체 859개


In [11]:

# ── IP 대조 ──────────────────────────────────────────────────────
print('\n=== IP 대조 ===')
if 'IP' not in df_reviewed.columns:
    print('IP 컬럼 없음 → 스킵')
else:
    ip_flags  = df_reviewed['IP'].astype(str).str.strip().str.upper()
    ip_kws    = df_reviewed.loc[ip_flags == 'O', '나머지_키워드'].tolist()
    print(f'IP 표시 키워드: {len(ip_kws):,}개')

    df_ip          = pd.read_parquet(IP_KW_PATH)
    existing_names = set(df_ip['ip_name'].astype(str).tolist())

    already_in = [kw for kw in ip_kws if kw in existing_names]
    new_ip     = [kw for kw in ip_kws if kw not in existing_names]

    print(f'\n  기존 ip_keywords에 있는 IP ({len(already_in)}개):')
    for ip in already_in:
        print(f'    ✓ {ip}')

    print(f'\n  신규 IP — ip_keywords에 없음 ({len(new_ip)}개):')
    for ip in new_ip:
        print(f'    ✗ {ip}')


=== IP 대조 ===
IP 표시 키워드: 43개

  기존 ip_keywords에 있는 IP (19개):
    ✓ 이정후
    ✓ 부창제과
    ✓ 테디베어
    ✓ 헬로키티
    ✓ 좀비딸
    ✓ 히밥
    ✓ 뵈르뵈르
    ✓ 미노리키친
    ✓ 이장우
    ✓ 라인프렌즈
    ✓ 티처스
    ✓ 김잼작가
    ✓ 아티제
    ✓ 마이멜로디
    ✓ 주토피아
    ✓ 이봉원
    ✓ 하정우
    ✓ 장민호
    ✓ SK하이닉스

  신규 IP — ip_keywords에 없음 (24개):
    ✗ K리그
    ✗ 맛장우
    ✗ 기사식당
    ✗ 황인택
    ✗ 대웅제약
    ✗ 한도초과
    ✗ 세븐셀렉트
    ✗ 바른목장
    ✗ 정희원교수님
    ✗ 장충동
    ✗ 총동원
    ✗ 민규
    ✗ APEC정상회의공식협찬사
    ✗ 장수회관
    ✗ 부르봉
    ✗ 애용이
    ✗ 푸하하
    ✗ 마루와유지
    ✗ 더기와
    ✗ 도쿄브레드
    ✗ 피카츄
    ✗ 잠바주스
    ✗ 정승제
    ✗ 김부장이야기


## Phase 9. 신규 IP 키워드 추가

07 Phase 5에서 식별된 신규 IP 13개를 `ip_keywords.parquet`에 추가합니다.

In [12]:
IP_KW_PATH = os.path.join(PROC_DIR, 'ip_keywords.parquet')

# ── 신규 IP 키워드 정의 ───────────────────────────────────────────
new_ip_raw = {
    '피카츄':              ['캐릭터', '포켓몬', '귀여움', '애니메이션', '콜라보', '한정판'],
    '애용이':              ['캐릭터', '웹툰', '고양이', '귀여움', '콜라보', '좀비딸'],
    '마루와유지':          ['일본', '식품', '수입', '브랜드', '글로벌', '크림', '브륄레', '스프레드'],
    '잠바주스':            ['주스', '스무디', '과일', '건강', '프랜차이즈', '글로벌'],
    '더기와':              ['주점', '전통주', '안주', '퓨전', '한식', '감성'],
    '기사식당':            ['한식', '가정식', '식당', '일상', '집밥'],
    '장수회관':            ['전통', '노포', '한식', '맛집', '콜라보'],
    '바른목장':            ['유제품', '우유', '신선', '목장', '건강', '자연'],
    '총동원':              ['참치', '수산', '동원', 'PB', '브랜드'],
    '부르봉':              ['일본', '제과', '쿠키', '비스킷', '수입'],
    '도쿄브레드':          ['일본', '베이커리', '빵', '수입', '도쿄'],
    '대웅제약':            ['건강', '제약', '비타민', '건강기능식품', '브랜드'],
    'APEC정상회의공식협찬사': ['공식', '글로벌', '한정판', '행사', '마케팅'],
}

# ── run_pipeline() 정규화 적용 ────────────────────────────────────
new_ip_processed = {
    ip_name: run_pipeline(kws)
    for ip_name, kws in new_ip_raw.items()
}

# ── 기존 ip_keywords.parquet 로드 후 중복 체크 ───────────────────
df_ip = pd.read_parquet(IP_KW_PATH)
existing_names = set(df_ip['ip_name'].astype(str).tolist())

already = [ip for ip in new_ip_processed if ip in existing_names]
to_add  = {ip: kws for ip, kws in new_ip_processed.items() if ip not in existing_names}

if already:
    print(f'이미 존재 (스킵): {already}')

print(f'추가 대상: {len(to_add)}개')
for ip, kws in to_add.items():
    print(f'  {ip}: {kws}')

# ── append 후 저장 ────────────────────────────────────────────────
df_new = pd.DataFrame([
    {'ip_name': ip, '키워드': kws}
    for ip, kws in to_add.items()
])

df_ip_updated = pd.concat([df_ip, df_new], ignore_index=True)
df_ip_updated.to_parquet(IP_KW_PATH, index=False)

print(f'\n저장 완료: {IP_KW_PATH}')
print(f'기존 {len(df_ip)}개 → 업데이트 후 {len(df_ip_updated)}개')

추가 대상: 13개
  피카츄: ['캐릭터', '포켓몬스터', '귀여움', '애니메이션', '콜라보', '한정판매']
  애용이: ['캐릭터', '웹툰', '고양이', '귀여움', '콜라보', '좀비딸']
  마루와유지: ['일본', '식품', '수입', '브랜드', '글로벌', '크림', '브륄레', '스프레드']
  잠바주스: ['주스', '스무디', '과일', '건강', '프랜차이즈', '글로벌']
  더기와: ['주점', '전통주', '안주', '퓨전', '한식', '감성']
  기사식당: ['한식', '가정식', '식당', '일상', '집밥']
  장수회관: ['전통', '노포', '한식', '맛집', '콜라보']
  바른목장: ['유제품', '우유', '신선함', '목장', '건강', '자연']
  총동원: ['참치', '수산', '동원', 'PB', '브랜드']
  부르봉: ['일본', '제과', '쿠키', '비스킷', '수입']
  도쿄브레드: ['일본', '베이커리', '빵', '수입', '도쿄']
  대웅제약: ['건강', '제약', '비타민', '건강기능식품', '브랜드']
  APEC정상회의공식협찬사: ['공식', '글로벌', '한정판매', '행사', '마케팅']

저장 완료: c:\Users\송정현\Documents\Projects\박재홍교수님세미나\Projects\20기\7eleven_npd_framework\data\processed\ip_keywords.parquet
기존 275개 → 업데이트 후 288개


## Phase 6. 제품별 최종 키워드셋 확정

Phase 5까지 적용된 결과 기준:
- `seven_remaining_kw_review_final.xlsx`가 있으면 O/X 반영 후 `final_df['풀_교집합']` 사용
- 없으면 `result_df['확정_키워드']` (run_pipeline 적용본) 사용


In [13]:
if 'final_df' in dir():
    # Phase 5 처리까지 반영
    kw_final = final_df[['제품명', '최종키워드']].copy()
else:
    # Phase 5 미실행 — run_pipeline 적용본 사용
    kw_final = (
        result_df[['제품명', '확정_키워드']]
        .rename(columns={'확정_키워드': '최종키워드'})
        .copy()
    )

has_kw = kw_final['최종키워드'].apply(lambda x: isinstance(x, list) and len(x) > 0)
print(f'제품 수: {len(kw_final)}개')
print(f'키워드 있는 제품: {has_kw.sum()}개 / 없는 제품: {(~has_kw).sum()}개')
print()
print(kw_final[has_kw].head(5).to_string())

제품 수: 859개
키워드 있는 제품: 847개 / 없는 제품: 12개

          제품명                       최종키워드
0  25년 꿀고구마호빵            [가을, 달콤, 제철, 호빵]
1  25년 듬뿍피자호빵       [가을, 제철, 토마토, 피자, 호빵]
2  25년 생생야채호빵        [가을, 고기, 야채, 제철, 호빵]
3  25년 정통단팥호빵         [가을, 팥, 달콤, 제철, 호빵]
4   2분 삼겹김치찌개  [겨울, 김치, 도시락, 삼겹살, 야식, 찌개]


## Phase 7. engagement parquet 키워드 업데이트 준비

`instagram_engagement_with_keywords.parquet` 세븐일레븐 행 중
`kw_final['제품명']`과 `원본명` 기준으로 매칭하여 키워드 업데이트 맵을 구성합니다.
(원본명 → 정규화명 → 최종키워드)

In [14]:
import re
from difflib import get_close_matches

INSTA_PQ = os.path.join(BASE_DIR, 'data', 'processed', 'instagram_engagement_with_keywords.parquet')
insta    = pd.read_parquet(INSTA_PQ)
seven    = insta[insta['편의점명'] == '세븐일레븐'].copy()

def norm_name(s):
    """공백·특수문자 제거 + 소문자 → 매칭 키"""
    return re.sub(r'[\s!&\(\)\[\].·]', '', str(s)).lower()

# kw_final 제품명 → 정규화 키 매핑
xl_map   = {norm_name(row['제품명']): row['제품명'] for _, row in kw_final.iterrows()}

# engagement 원본명 → 정규화 키 매핑 (원본명 기준)
orig_map = {}
for orig in seven['원본명'].dropna().unique():
    key = norm_name(orig)
    orig_map.setdefault(key, orig)

# 매칭
matched_keys   = set(xl_map) & set(orig_map)
unmatched_orig = set(orig_map) - set(xl_map)

print(f'kw_final 제품 수:          {len(xl_map):,}개')
print(f'engagement 원본명 unique:  {len(orig_map):,}개')
print(f'매칭 (원본명 기준):         {len(matched_keys):,}개')
print(f'미매칭 (원본명 기준):       {len(unmatched_orig):,}개')

# 미매칭 → 게시물 수 + difflib 후보
post_cnt = (
    seven[seven['원본명'].apply(norm_name).isin(unmatched_orig)]
    .groupby('원본명').size().rename('게시물수')
)
xl_keys = list(xl_map.keys())
rows = []
for key in sorted(unmatched_orig):
    orig_name = orig_map[key]
    cnt   = post_cnt.get(orig_name, 0)
    close = get_close_matches(key, xl_keys, n=1, cutoff=0.6)
    best  = xl_map[close[0]] if close else ''
    rows.append({'원본명': orig_name, '게시물수': cnt, 'Excel_유사후보': best})

unmatch_df = pd.DataFrame(rows).sort_values('게시물수', ascending=False).reset_index(drop=True)

UNMATCH_OUT = os.path.join(EDA_DIR, 'seven_insta_unmatched_products.csv')
unmatch_df.to_csv(UNMATCH_OUT, index=False, encoding='utf-8-sig')
print(f'\n미매칭 CSV 저장: {UNMATCH_OUT}')
unmatch_df

kw_final 제품 수:          798개
engagement 원본명 unique:  791개
매칭 (원본명 기준):         765개
미매칭 (원본명 기준):       26개

미매칭 CSV 저장: c:\Users\송정현\Documents\Projects\박재홍교수님세미나\Projects\20기\7eleven_npd_framework\eda\seven_insta_unmatched_products.csv


,원본명,게시물수,Excel_유사후보
0,카페라떼,3,
1,깐부도시락(깐풍기&마파두부),2,
2,바닐라라떼,2,테디베어 바닐라빈 라떼
3,카라멜라떼,2,
4,테디베어 미니케이크(초코),2,테디베어 미니케이크
5,테디베어 미니케이크(우유),2,테디베어 미니케이크
6,메론킥,2,
7,고기(肉)올인원도시락,1,고기올인원도시락
8,7-SELECT 젤리초코볼 오렌지,1,7 SELECT 젤리초코볼 오렌지
9,7-SELECT 젤리초코볼 샤인머스켓,1,7 SELECT 젤리초코볼 샤인머스켓


## Phase 8. instagram_engagement_with_keywords.parquet 키워드 업데이트

1. 매칭된 원본명(829개) → `kw_final['최종키워드']`로 업데이트
2. 미매칭 원본명(26개) → `seven_insta_unmatched_products_final.csv`의 `Excel_유사후보` 키워드로 업데이트
3. 그래도 매핑 없는 행 → 기존 키워드 유지

In [15]:
UNMATCH_FINAL = os.path.join(EDA_DIR, 'seven_insta_unmatched_products_final.csv')
UNMATCH_AUTO  = os.path.join(EDA_DIR, 'seven_insta_unmatched_products.csv')
ENGAGEMENT_PQ  = os.path.join(PROC_DIR, 'instagram_engagement_with_keywords.parquet')
ENGAGEMENT_CSV = os.path.join(PROC_DIR, 'instagram_engagement_with_keywords.csv')

insta_eng = pd.read_parquet(ENGAGEMENT_PQ)
seven_eng = insta_eng[insta_eng['편의점명'] == '세븐일레븐'].copy()
print(f'engagement 로드: {len(insta_eng):,}행 (세븐일레븐 {len(seven_eng):,}행)')

# kw_final 룩업: norm(제품명) → 최종키워드
kw_lookup = {norm_name(row['제품명']): row['최종키워드'] for _, row in kw_final.iterrows()}

# ── Step 1. 매칭된 원본명 업데이트 맵 ─────────────────────────────
update_map = {}
for orig in seven_eng['원본명'].dropna().unique():
    key = norm_name(orig)
    if key in kw_lookup:
        update_map[orig] = kw_lookup[key]

print(f'Step 1. 매칭 원본명 업데이트: {len(update_map):,}개')

# ── Step 2. 미매칭 원본명 처리 (Excel_유사후보) ───────────────────
unmatch_src = UNMATCH_FINAL if os.path.exists(UNMATCH_FINAL) else UNMATCH_AUTO
df_unmatch  = pd.read_csv(unmatch_src)

# 컬럼명 통일: 구버전 '정규화명' → '원본명'
if '원본명' not in df_unmatch.columns and '정규화명' in df_unmatch.columns:
    df_unmatch = df_unmatch.rename(columns={'정규화명': '원본명'})

print(f'Step 2. 미매칭 파일: {os.path.basename(unmatch_src)} ({len(df_unmatch)}행)')

cnt_applied, cnt_skipped = 0, 0
for _, row in df_unmatch.iterrows():
    orig = str(row['원본명']).strip()
    best = str(row['Excel_유사후보']).strip()
    if best and best.lower() != 'nan':
        key = norm_name(best)
        if key in kw_lookup:
            update_map[orig] = kw_lookup[key]
            cnt_applied += 1
            continue
    cnt_skipped += 1

print(f'  유사후보 적용: {cnt_applied}개 / 스킵(후보없음): {cnt_skipped}개')
print(f'  최종 업데이트 맵: {len(update_map):,}개 원본명')

# ── Step 3. 키워드_정제 컬럼 추가 ────────────────────────────────
insta_updated = insta_eng.copy()

insta_updated['키워드_정제'] = insta_updated.apply(
    lambda row: update_map.get(str(row['원본명'])) if row['편의점명'] == '세븐일레븐' else None,
    axis=1
)

mask_seven   = insta_updated['편의점명'] == '세븐일레븐'
updated_mask = mask_seven & insta_updated['원본명'].isin(update_map)
kept_mask    = mask_seven & ~insta_updated['원본명'].isin(update_map)

print(f'\nStep 3. 세븐일레븐 행: {mask_seven.sum():,}행')
print(f'  키워드_정제 채워짐: {updated_mask.sum():,}행')
print(f'  키워드_정제 없음(None): {kept_mask.sum():,}행')

kept_origs = insta_updated.loc[kept_mask, '원본명'].dropna().unique().tolist()
if kept_origs:
    print(f'\n  키워드_정제 없는 원본명 ({len(kept_origs)}개):')
    for n in sorted(kept_origs):
        print(f'    - {n}')

# ── Step 4. 키워드_개수 컬럼 추가 ────────────────────────────────
def count_keywords(row):
    if row['편의점명'] == '세븐일레븐':
        kws = row['키워드_정제']
    else:
        kws = row['키워드']
    if isinstance(kws, list):
        return len(kws)
    return 0

insta_updated['키워드_개수'] = insta_updated.apply(count_keywords, axis=1)

print(f'\nStep 4. 키워드_개수 컬럼 추가 완료')
print(insta_updated.groupby('편의점명')['키워드_개수'].describe().round(1))

# ── Step 5. 저장 (parquet + CSV) ─────────────────────────────────
insta_updated.to_parquet(ENGAGEMENT_PQ, index=False)

csv_df = insta_updated.copy()
csv_df['키워드']    = csv_df['키워드'].apply(lambda x: str(x) if isinstance(x, list) else x)
csv_df['키워드_정제'] = csv_df['키워드_정제'].apply(lambda x: str(x) if isinstance(x, list) else x)
csv_df.to_csv(ENGAGEMENT_CSV, index=False, encoding='utf-8-sig')

print(f'\n저장 완료')
print(f'  parquet : {ENGAGEMENT_PQ}')
print(f'  CSV     : {ENGAGEMENT_CSV}')

engagement 로드: 4,249행 (세븐일레븐 1,301행)
Step 1. 매칭 원본명 업데이트: 829개
Step 2. 미매칭 파일: seven_insta_unmatched_products.csv (26행)
  유사후보 적용: 15개 / 스킵(후보없음): 11개
  최종 업데이트 맵: 844개 원본명

Step 3. 세븐일레븐 행: 1,301행
  키워드_정제 채워짐: 1,285행
  키워드_정제 없음(None): 16행

  키워드_정제 없는 원본명 (11개):
    - 깐부도시락(깐풍기&마파두부)
    - 누네띠네 딥초코
    - 뉴룽지 카라멜 크룽지맛
    - 메론킥
    - 선양소주
    - 아이스 라떼
    - 주토피아구슬젤리
    - 치즈오븐스파게티
    - 카라멜라떼
    - 카페라떼
    - 하겐다즈 파인트

Step 4. 키워드_개수 컬럼 추가 완료
        count  mean  std  min  25%  50%  75%   max
편의점명                                              
CU     1459.0   0.0  0.0  0.0  0.0  0.0  0.0   0.0
GS25   1489.0   0.0  0.0  0.0  0.0  0.0  0.0   0.0
세븐일레븐  1301.0   7.3  5.5  0.0  4.0  6.0  8.0  30.0

저장 완료
  parquet : c:\Users\송정현\Documents\Projects\박재홍교수님세미나\Projects\20기\7eleven_npd_framework\data\processed\instagram_engagement_with_keywords.parquet
  CSV     : c:\Users\송정현\Documents\Projects\박재홍교수님세미나\Projects\20기\7eleven_npd_framework\data\processed\instagram_engagement_with_keywords.csv